# CPP 接入 ZEUS 向量

完整图像复现阶段暂时跳过。本 notebook 直接读取已保存的 ZEUS 向量，先运行 OpenML ID 61 和 ID 14，再扩展到全部 34 个数据集。

输入使用 `embeddings_scaled`，保持现有 ZEUS + K-means 的输入条件。每个数据集独立训练 CPP；真实标签只用于确定预测时的组数和计算 ARI。ZEUS 权重与向量保持固定。

使用官方 `CPPNet`、`SinkhornDistance`、`MLCLoss` 和 `TotalCodingRate`。输入层由 768 改为 512；初始设置为隐藏维度 4096、输出维度 128、批量 128、训练 5 轮，其中第一轮用于初始化。这是预先固定的表格数据试跑设置，不是经过逐数据集调优的设置。

In [1]:
from pathlib import Path
from subprocess import check_output
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score
from torch.utils.data import DataLoader, TensorDataset

root = Path.cwd().parent
cpp_dir = root / "vendor" / "CPP"
sys.path.insert(0, str(cpp_dir))

from model.CPP_model import CPPNet
from model.sink_distance import SinkhornDistance
from loss.loss_fn import MLCLoss, TotalCodingRate

cpp_commit = check_output(
    ["git", "-c", f"safe.directory={cpp_dir.as_posix()}", "-C", str(cpp_dir), "rev-parse", "HEAD"],
    text=True,
).strip()
assert cpp_commit == "cb39bdc65f346b4cb5950d14af2e393a5f6a3566"

torch.set_num_threads(4)
device = torch.device("cpu")
config = {"seed": 42, "hidden_dim": 4096, "z_dim": 128, "batch_size": 128,
          "epochs": 5, "lr": 1e-4, "sinkhorn_temperature": 0.175, "sinkhorn_iterations": 5}

baseline = pd.read_csv(root / "results" / "zeus_openml_baseline.csv")
baseline = baseline.loc[baseline["method"] == "scaled_zeus"].set_index("openml_id")

prediction_dir = root / "data" / "cpp_zeus"
model_dir = root / "models" / "cpp_zeus"
prediction_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__, "device:", device, "CPP commit:", cpp_commit)
print(config)

PyTorch: 2.5.1+cpu device: cpu CPP commit: cb39bdc65f346b4cb5950d14af2e393a5f6a3566
{'seed': 42, 'hidden_dim': 4096, 'z_dim': 128, 'batch_size': 128, 'epochs': 5, 'lr': 0.0001, 'sinkhorn_temperature': 0.175, 'sinkhorn_iterations': 5}


## 训练与预测

第一轮只训练特征分支，再将其权重复制给关联分支；后四轮共同训练。数据读取、训练和预测分别使用简单函数，后续接入更多数据集时复用这些函数。

预测在完整数据集上计算样本关联，将关联矩阵对称化后调用 scikit-learn 的谱聚类，固定随机种子，K-means 初始化次数为 100。对称化用于提供谱聚类需要的对称关联矩阵；这是本项目预测入口与官方批次评估入口的实现差异。

In [2]:
def train_cpp(features):
    torch.manual_seed(config["seed"])
    np.random.seed(config["seed"])
    model = CPPNet(features.shape[1], config["hidden_dim"], config["z_dim"]).to(device)
    loader = DataLoader(TensorDataset(features), batch_size=config["batch_size"],
                        shuffle=True, drop_last=False)
    sinkhorn = SinkhornDistance(config["sinkhorn_temperature"], config["sinkhorn_iterations"])
    warmup_loss = TotalCodingRate(eps=0.1)
    cpp_loss = MLCLoss(eps=0.1, gamma=1.0)
    feature_optimizer = torch.optim.SGD(
        list(model.pre_feature.parameters()) + list(model.subspace.parameters()),
        lr=config["lr"], momentum=0.9, weight_decay=1e-4)
    cluster_optimizer = torch.optim.SGD(
        model.cluster.parameters(), lr=config["lr"], momentum=0.9, weight_decay=5e-3)

    history = []
    started = time.perf_counter()
    for epoch in range(config["epochs"]):
        model.train()
        losses = []
        for (x,) in loader:
            x = x.to(device)
            z, codes = model(x)
            if epoch == 0:
                loss = warmup_loss(z)
            else:
                membership = sinkhorn((codes @ codes.T).abs()[None])[0][0] * len(x)
                loss, _ = cpp_loss(z, membership)
                loss = loss + 0.05 * 0.5 * membership.norm().square()
            assert torch.isfinite(loss), "Non-finite CPP loss"
            feature_optimizer.zero_grad()
            cluster_optimizer.zero_grad()
            loss.backward()
            feature_optimizer.step()
            if epoch > 0:
                cluster_optimizer.step()
            losses.append(loss.item())
        if epoch == 0:
            model.cluster.load_state_dict(model.subspace.state_dict())
        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses))})
    return model, history, time.perf_counter() - started


def predict_cpp(model, features, n_clusters):
    model.eval()
    sinkhorn = SinkhornDistance(config["sinkhorn_temperature"], config["sinkhorn_iterations"])
    with torch.inference_mode():
        _, codes = model(features.to(device))
        membership = sinkhorn((codes @ codes.T).abs()[None])[0][0] * len(features)
        affinity = ((membership + membership.T) / 2).cpu().numpy()
    return SpectralClustering(
        n_clusters=n_clusters, affinity="precomputed",
        random_state=config["seed"], n_init=100,
    ).fit_predict(affinity)

In [3]:
def run_dataset(openml_id):
    with np.load(root / "data" / "embeddings" / f"openml_{openml_id}.npz") as data:
        features = torch.from_numpy(data["embeddings_scaled"].astype(np.float32))
        labels = data["labels"].copy()
        sample_indices = data["sample_indices"].copy()
    assert features.shape == (len(labels), 512) and torch.isfinite(features).all()
    n_clusters = len(np.unique(labels))
    model, history, training_seconds = train_cpp(features)
    started = time.perf_counter()
    predicted = predict_cpp(model, features, n_clusters)
    prediction_seconds = time.perf_counter() - started
    ari = adjusted_rand_score(labels, predicted)

    np.savez(prediction_dir / f"openml_{openml_id}.npz",
             predicted=predicted, labels=labels, sample_indices=sample_indices)
    torch.save({"model": model.state_dict(), "config": config, "cpp_commit": cpp_commit,
                "openml_id": openml_id, "embedding_variant": "scaled"},
               model_dir / f"openml_{openml_id}.pt")
    pd.DataFrame(history).to_csv(prediction_dir / f"openml_{openml_id}_training.csv", index=False)

    row = {"openml_id": openml_id, "n_samples": len(labels), "input_dim": 512,
           "requested_k": n_clusters, "predicted_k": len(np.unique(predicted)),
           "ari": ari, "kmeans_ari": float(baseline.loc[openml_id, "ari"]),
           "training_seconds": training_seconds, "prediction_seconds": prediction_seconds,
           "seed": config["seed"], "batch_size": config["batch_size"],
           "epochs": config["epochs"], "embedding_variant": "scaled", "cpp_commit": cpp_commit}
    row["delta_vs_kmeans"] = row["ari"] - row["kmeans_ari"]
    print(f"ID {openml_id}: ARI {ari:.4f}, K-means {row['kmeans_ari']:.4f}, "
          f"train {training_seconds:.1f}s, predict {prediction_seconds:.1f}s", flush=True)
    return row

## 先运行两个数据集

ID 61 和 ID 14 分别覆盖小样本和较大样本。检查每个样本都有预测，预测顺序与缓存样本索引一致，并保存训练参数和预测。

In [4]:
rows = [run_dataset(openml_id) for openml_id in (61, 14)]
smoke_results = pd.DataFrame(rows)
smoke_results.to_csv(root / "results" / "cpp_zeus_smoke.csv", index=False)
display(smoke_results[["openml_id", "n_samples", "ari", "kmeans_ari", "delta_vs_kmeans"]])

ID 61: ARI 0.7323, K-means 0.8515, train 1.0s, predict 0.6s


ID 14: ARI 0.4433, K-means 0.5056, train 9.9s, predict 1.8s


,openml_id,n_samples,ari,kmeans_ari,delta_vs_kmeans
0,61,150,0.732298,0.851457,-0.119159
1,14,2000,0.443319,0.505635,-0.062315


## 扩展到全部 34 个数据集

继续使用同一组训练参数。完成后汇总全部数据集的 ARI，并与同一份 ZEUS 向量上的 K-means 比较。逐数据集的预测和训练记录位于 `data/cpp_zeus/`，模型位于 `models/cpp_zeus/`。

In [5]:
openml_ids = sorted(int(path.stem.split("_")[1])
                    for path in (root / "data" / "embeddings").glob("openml_*.npz"))
for openml_id in openml_ids:
    if openml_id not in (61, 14):
        rows.append(run_dataset(openml_id))
    results = pd.DataFrame(rows).sort_values("openml_id")
    results.to_csv(root / "results" / "cpp_zeus_openml.csv", index=False)

display(results[["ari", "kmeans_ari"]].mean().to_frame("mean ARI"))
print("Datasets:", len(results), "CPP wins:", int((results["delta_vs_kmeans"] > 1e-6).sum()))

ID 15: ARI 0.8127, K-means 0.8128, train 3.5s, predict 0.7s


ID 16: ARI 0.5892, K-means 0.7403, train 9.6s, predict 1.9s


ID 18: ARI 0.4847, K-means 0.5156, train 10.1s, predict 1.9s


ID 22: ARI 0.5031, K-means 0.5607, train 9.6s, predict 1.8s


ID 35: ARI 0.8616, K-means 0.8512, train 1.8s, predict 0.6s


...

ID 40705: ARI 0.3818, K-means 0.4545, train 4.7s, predict 0.9s


ID 42261: ARI 0.7323, K-means 0.8515, train 1.0s, predict 0.4s


ID 42585: ARI 0.9505, K-means 0.9505, train 1.8s, predict 0.5s


,mean ARI
ari,0.498960
kmeans_ari,0.573642


Datasets: 34 CPP wins: 12


## 与现有 LAPD 比较

LAPD 的“提供 K”实验使用了类别数，但实际输出组数未必等于 K。比较均值时使用 CPP、LAPD 都有完整预测的同一批数据集，并报告数量。当前只比较提供类别数的实验条件。

In [6]:
lapd = pd.read_csv(root / "results" / "lapd_stage4_audit.csv")
lapd = lapd.loc[(lapd["mode"] == "known") & (lapd["status"] == "complete"),
                ["openml_id", "ari"]].rename(columns={"ari": "lapd_ari"})
matched = results.merge(lapd, on="openml_id")
print("Matched complete datasets:", len(matched))
display(matched[["ari", "kmeans_ari", "lapd_ari"]].mean().to_frame("mean ARI"))

Matched complete datasets: 27


,mean ARI
ari,0.510208
kmeans_ari,0.600688
lapd_ari,0.326491


## 本次运行结果

34 个数据集都已完成。全体平均 ARI：CPP **0.499**，ZEUS + K-means **0.574**。CPP 在 12 个数据集上更高，6 个近似相同，16 个更低；比较时将绝对差异不超过 `1e-6` 视为近似相同。

在 LAPD 的“提供 K”实验有完整预测的同一批 27 个数据集上，平均 ARI 为：CPP **0.510**，ZEUS + K-means **0.601**，LAPD **0.326**。

全部训练耗时约 95 秒，预测约 24 秒。预测数量、标签与样本索引顺序均已核对。当前为固定参数、单随机种子的初始实验，完整图像复现仍暂时跳过。